# Detecção de Fraudes em Transaçaões

In [1]:
import pandas as pd

url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'

df = pd.read_csv(url)

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# **Problema de classificação desbalanceada**

Fraudes sao raras -> modelo pode ignorar a classe 1

In [2]:
#calculando a proporção de cada tipo de tansação. Coluna class do dataset - 0 normal / 1 - fraude
df['Class'].value_counts(normalize=True)

,proportion
Class,
0,0.998273
1,0.001727


O resultado acima, mostra que 98% das transaçoes são normais e 0,17% são fraudes. Issmo mostra que o dataset é desbalanceado, isso que dizer, quando uma classe apaece mais que a outra.

# Feature Engineering

In [3]:
import numpy as np

#diminuir escala de valores muito grandes e para evitar valores com log 0.
#Quando aplicamos logaritimo ajuda o modelo a indentificar padrões mais corretamente
df['Amount_log'] = np.log1p(df['Amount'])


In [4]:
#permitir comparação com valres muito diferentes entre si.
#Forma de garantir que as variaveis do moelo tenham o mesmo peso
from sklearn.preprocessing import StandardScaler

skaler = StandardScaler()

df['Amount_scaled'] = skaler.fit_transform(df[['Amount']])

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V23,V24,V25,V26,V27,V28,Amount,Class,Amount_log,Amount_scaled
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0,5.014760,0.244964
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0,1.305626,-0.342475
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,5.939276,1.160686
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0,4.824306,0.140534
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0,4.262539,-0.073403


### 1. Preparação dos Dados com Divisão Estratificada (`stratify`)

In [5]:
from sklearn.model_selection import train_test_split

# Definindo as features (incluindo as transformadas) e o target
# Vamos remover 'Amount' original e 'Amount_log' para evitar multicolinearidade extrema com o scaled,
# ou podemos manter a que preferir. Vamos usar as features recomendadas:
features = [col for col in df.columns if col not in ['Class', 'Amount', 'Amount_log']]
X = df[features]
y = df['Class']

# Divisão estratificada para manter a proporção exata de fraudes no treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 2. Treinamento do Baseline (Regressão Logística, Random Forest e XGBoost)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
print("=== LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_lr))

# 2. Random Forest
rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf))

# 3. XGBoost (usando scale_pos_weight para balanceamento)
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_model = XGBClassifier(scale_pos_weight=scale_pos, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("=== XGBOOST ===")
print(classification_report(y_test, y_pred_xgb))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== LOGISTIC REGRESSION ===
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962

=== RANDOM FOREST ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.79      0.83      0.81        98

    accuracy                           1.00     56962
   macro avg       0.90      0.91      0.90     56962
weighted avg       1.00      1.00      1.00     56962

=== XGBOOST ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.87      0.83      0.85        98

    accuracy                           1.00     56962
   macro avg       0.94      0.91      0.92     56962
weighted

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Criando o Pipeline com o StandardScaler e o RandomForestClassifier como exemplo balanceado
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

# Treinando o Pipeline
pipeline_rf.fit(X_train, y_train)

# Exibindo o Pipeline criado
print(pipeline_rf)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                        random_state=42))])


### 3. Avaliação Comparativa de Curvas (ROC e Precision-Recall)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# Probabilidades da classe positiva (fraude)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Plot ROC Curve
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
for label, prob in [('Logistic Regression', y_prob_lr), ('Random Forest', y_prob_rf), ('XGBoost', y_prob_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()

# Plot Precision-Recall Curve
plt.subplot(1, 2, 2)
for label, prob in [('Logistic Regression', y_prob_lr), ('Random Forest', y_prob_rf), ('XGBoost', y_prob_xgb)]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    plt.plot(rec, prec, label=f"{label} (AP = {auc(rec, prec):.3f})")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.legend()
plt.tight_layout()
plt.show()

### 4. Ajustes Finos: Pipeline, Limiar de Decisão Personalizado em 0.3 e GridSearchCV

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# Criando o Pipeline (utilizando o RandomForestClassifier como exemplo do Grid)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

# Definindo parâmetros de busca para o Grid
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10]
}

# Otimizando pelo Recall para focar na captura de fraudes
grid_search = GridSearchCV(pipeline, param_grid, scoring='recall', cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_
print(f"Melhores Parâmetros: {grid_search.best_params_}")

In [ ]:
# Modificando o limiar de decisão para 0.3
y_prob_best = best_pipeline.predict_proba(X_test)[:, 1]
y_pred_custom_threshold = (y_prob_best >= 0.3).astype(int)

print("=== RELATÓRIO COM LIMIAR CUSTOMIZADO (0.3) ===")
print(classification_report(y_test, y_pred_custom_threshold))

### 5. Explicação do Modelo: Importância das Variáveis e SHAP

In [ ]:
# Instalação do SHAP
!pip install shap -q

In [ ]:
import shap

# Feature Importance nativo do Random Forest otimizado
best_rf_model = best_pipeline.named_steps['classifier']
importances = best_rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 5))
plt.title("Importância das Variáveis (Random Forest)")
plt.bar(range(X_train.shape[1]), importances[indices], align="center")
plt.xticks(range(X_train.shape[1]), np.array(features)[indices], rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Usando SHAP para explicar as decisões locais e globais
# Como o dataset é grande, usaremos uma amostra para agilizar o cálculo
explainer = shap.TreeExplainer(best_rf_model)
shap_values = explainer.shap_values(X_test.iloc[:500])

# Gráfico de resumo das variáveis impactantes
shap.summary_plot(shap_values, X_test.iloc[:500])